# 84 — Free-Wilson / Matched Molecular Pair Scaffold Decomposition

**Classical QSAR wisdom:** in an analog series, pEC50 ≈ μ_scaffold + Σ(R_group_i contribution).

The Free-Wilson model is additive: each R-group substituent has a fixed contribution
to pEC50 regardless of what other R-groups are present. It works best when:
1. You have many analogs of the same scaffold ✓ (test = analog expansion)
2. The substituent effects are additive ✓ (common for PXR ligands)

Implementation:
1. Use RDKit MMP fragmentation to decompose each compound into scaffold + R-groups
2. Encode R-group identity as binary features (one-hot per unique R-group)
3. Fit ridge regression → each R-group gets a coefficient
4. For test compounds: look up their R-groups and predict from scaffold+R-group model


In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R²={r2:.4f} "
              f"r={pr:.4f} ρ={sp:.4f} τ={kt:.4f}{ca}")
    return m

In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 149


In [4]:
from rdkit import Chem
from rdkit.Chem import rdMMPA, Scaffolds
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict

def decompose_rgroups(smiles_list, max_cuts=1):
    """MMP single-cut decomposition: scaffold + R-group."""
    results = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            results.append({"scaffold": smi, "rgroups": []})
            continue
        try:
            frags = rdMMPA.FragmentMol(mol, maxCuts=max_cuts, resultsAsMols=False)
            # frags = list of (core, rgroup) SMILES pairs
            if not frags:
                scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
                results.append({"scaffold": scaffold, "rgroups": []})
                continue
            # Pick the fragmentation with the largest core
            best = max(frags, key=lambda x: Chem.MolFromSmiles(x[0]).GetNumHeavyAtoms()
                       if Chem.MolFromSmiles(x[0]) else 0)
            results.append({"scaffold": best[0], "rgroups": [best[1]] if best[1] else []})
        except:
            results.append({"scaffold": smi, "rgroups": []})
    return results

print("Decomposing training set into scaffold + R-groups...", flush=True)
decomp_tr = decompose_rgroups(tr["smiles"].tolist())
decomp_te = decompose_rgroups(te["smiles"].tolist())

# Count unique scaffolds and R-groups
scaffolds_freq = defaultdict(int)
rgroups_freq   = defaultdict(int)
for d in decomp_tr:
    scaffolds_freq[d["scaffold"]] += 1
    for rg in d["rgroups"]:
        rgroups_freq[rg] += 1

print(f"Unique scaffolds: {len(scaffolds_freq)}")
print(f"Unique R-groups:  {len(rgroups_freq)}")
print(f"Top scaffolds: {sorted(scaffolds_freq.items(), key=lambda x:-x[1])[:5]}")


Decomposing training set into scaffold + R-groups...


Unique scaffolds: 16
Unique R-groups:  4121
Top scaffolds: [('', 4123), ('c1ccccc1', 2), ('O=C1CNC(=O)CNC(=O)CNC(=O)CNC(=O)CNC(=O)CNC(=O)CNC(=O)CNC(=O)CNC(=O)CNC(=O)CN1', 1), ('O=C1CCO1', 1), ('c1ccc(CCCCNCc2ccc(-c3ccccn3)cc2)cc1', 1)]


In [5]:
# Keep scaffolds and R-groups that appear ≥ MIN_FREQ times
MIN_FREQ = 3
common_scaffolds = {s for s,c in scaffolds_freq.items() if c >= MIN_FREQ}
common_rgroups   = {r for r,c in rgroups_freq.items()   if c >= MIN_FREQ}
print(f"Common scaffolds (≥{MIN_FREQ}): {len(common_scaffolds)}")
print(f"Common R-groups (≥{MIN_FREQ}):  {len(common_rgroups)}")

scaffold_idx = {s:i for i,s in enumerate(sorted(common_scaffolds))}
rgroup_idx   = {r:i for i,r in enumerate(sorted(common_rgroups))}
n_s = len(scaffold_idx); n_r = len(rgroup_idx)

def build_fw_features(decomp_list):
    """Binary feature: scaffold present (n_s cols) + R-group present (n_r cols)."""
    X = np.zeros((len(decomp_list), n_s + n_r), dtype=np.float32)
    coverage = 0
    for i, d in enumerate(decomp_list):
        if d["scaffold"] in scaffold_idx:
            X[i, scaffold_idx[d["scaffold"]]] = 1; coverage += 1
        for rg in d["rgroups"]:
            if rg in rgroup_idx:
                X[i, n_s + rgroup_idx[rg]] = 1
    return X, coverage

X_fw_tr, cov_tr = build_fw_features(decomp_tr)
X_fw_te, cov_te = build_fw_features(decomp_te)
print(f"Training coverage: {cov_tr}/{len(tr)} ({100*cov_tr/len(tr):.1f}%)")
print(f"Test coverage: {cov_te}/{len(te)} ({100*cov_te/len(te):.1f}%)")
print(f"FW features: {X_fw_tr.shape}")


Common scaffolds (≥3): 1
Common R-groups (≥3):  0
Training coverage: 4123/4139 (99.6%)
Test coverage: 513/513 (100.0%)
FW features: (4139, 1)


In [6]:
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Model 1: Pure Free-Wilson (Ridge on scaffold+R-group binary features)
print("\n=== Free-Wilson model (Ridge regression) ===", flush=True)
oof_fw = np.full(len(y_tr), np.nan)
for fold, (tr_idx, va_idx) in enumerate(splits):
    fw = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0])
    fw.fit(X_fw_tr[tr_idx], y_tr[tr_idx])
    oof_fw[va_idx] = fw.predict(X_fw_tr[va_idx])
m_fw = full_metrics(y_tr, oof_fw, cliff_pairs, "free_wilson")
m_fw_a = full_metrics(y_tr[active_mask], oof_fw[active_mask], "fw [active]")

# Model 2: FW features concatenated with 2D Morgan (hybrid)
X_hybrid_tr = np.hstack([X_tr, X_fw_tr])
X_hybrid_te  = np.hstack([X_te, X_fw_te])
oof_hybrid = np.full(len(y_tr), np.nan)
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = lgb.train(LGBM, lgb.Dataset(X_hybrid_tr[tr_idx], label=y_tr[tr_idx]),
                  valid_sets=[lgb.Dataset(X_hybrid_tr[va_idx], label=y_tr[va_idx])],
                  callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_hybrid[va_idx] = m.predict(X_hybrid_tr[va_idx])
m_hyb = full_metrics(y_tr, oof_hybrid, cliff_pairs, "lgbm_fw_hybrid")

oof = oof_hybrid
print("\n" + pd.DataFrame([m_fw, m_fw_a, m_hyb],
      index=["free_wilson","fw_active","hybrid"]).round(4).to_string())

# R-group importance from Ridge (Free-Wilson coefficients)
fw_final = RidgeCV(alphas=[0.01,0.1,1.0,10.0,100.0]).fit(X_fw_tr, y_tr)
rg_coefs = fw_final.coef_[n_s:]
top_pos = sorted(zip(common_rgroups, rg_coefs), key=lambda x:-x[1])[:10]
top_neg = sorted(zip(common_rgroups, rg_coefs), key=lambda x:x[1])[:10]
print("\nTop activity-enhancing R-groups:")
for rg, c in top_pos: print(f"  {rg[:40]:40s} +{c:.3f}")
print("Top activity-reducing R-groups:")
for rg, c in top_neg: print(f"  {rg[:40]:40s} {c:.3f}")



=== Free-Wilson model (Ridge regression) ===


  [free_wilson] RAE=1.0011 MAE=0.9108 R²=-0.0035 r=-0.0232 ρ=-0.0696 τ=-0.0511  Cliff=nan


  [lgbm_fw_hybrid] RAE=0.5636 MAE=0.5128 R²=0.6033 r=0.7767 ρ=0.7285 τ=0.5361  Cliff=nan

                RAE     MAE       R2  Pearson  Spearman  Kendall  Cliff_acc
free_wilson  1.0011  0.9108  -0.0035  -0.0232   -0.0696  -0.0511        NaN
fw_active    7.0889  1.4865 -27.9463  -0.0149   -0.0853  -0.0635        NaN
hybrid       0.5636  0.5128   0.6033   0.7767    0.7285   0.5361        NaN

Top activity-enhancing R-groups:
Top activity-reducing R-groups:


In [7]:
m_final = lgb.train(LGBM, lgb.Dataset(X_hybrid_tr, label=y_tr), callbacks=[lgb.log_evaluation(-1)])
te_preds = np.clip(m_final.predict(X_hybrid_te), y_tr.min()-0.5, y_tr.max()+0.5)
# Also save FW features for use in grand ensemble
np.save(DATA_PROCESSED/"X_fw_tr.npy", X_fw_tr)
np.save(DATA_PROCESSED/"X_fw_te.npy", X_fw_te)
np.save(DATA_PROCESSED/"oof_free_wilson.npy", oof)
np.save(DATA_PROCESSED/"te_oof_free_wilson.npy", te_preds)
sub = pd.DataFrame({"Molecule Name":te["name"].values,"pEC50":te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"84_free_wilson_hybrid.csv"; sub.to_csv(p,index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")


Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\84_free_wilson_hybrid.csv
Test: min=2.06 med=4.96 max=5.95
